# 03 — Results Analysis (Classic vs CNN)

**Runtime:** local. Reads `reports/comparison.csv` produced by
`python -m src.evaluation.evaluate`.

## The story
All three models are scored on the **same** held-out test split (150 clips).
The headline result: **classic ML (SVM on librosa features) beats the
from-scratch CNN** on GTZAN — and is ~80x cheaper to run. This notebook backs
that up with the per-class breakdown and confusion matrices.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import image as mpimg

df = pd.read_csv(ROOT / 'reports' / 'comparison.csv', index_col=0)
df.round(3)

## Accuracy / macro-F1 / latency

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df[['accuracy', 'macro_f1']].plot(kind='bar', ax=ax[0], ylim=(0, 1))
ax[0].set_title('Accuracy & macro-F1 (higher = better)'); ax[0].set_xticklabels(df.index, rotation=0)
df['latency_ms'].plot(kind='bar', ax=ax[1], color='salmon')
ax[1].set_title('Inference latency (ms/clip, CPU — lower = better)'); ax[1].set_xticklabels(df.index, rotation=0)
plt.tight_layout(); plt.show()

## Per-class F1 (where each model wins or struggles)

In [ ]:
per_class = df[[c for c in df.columns if c.startswith('f1_')]]
per_class.columns = [c.replace('f1_', '') for c in per_class.columns]
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(per_class.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(per_class.shape[1])); ax.set_xticklabels(per_class.columns, rotation=45, ha='right')
ax.set_yticks(range(len(per_class.index))); ax.set_yticklabels(per_class.index)
for i in range(per_class.shape[0]):
    for j in range(per_class.shape[1]):
        ax.text(j, i, f'{per_class.values[i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
fig.colorbar(im, ax=ax, label='per-class F1'); ax.set_title('Per-class F1 by model')
plt.tight_layout(); plt.show()

## Confusion matrices (test set)

In [ ]:
imgs = [('svm', 'cm_svm.png'), ('xgboost', 'cm_xgboost.png'), ('cnn', 'cm_cnn.png')]
imgs = [(n, ROOT / 'reports' / f) for n, f in imgs if (ROOT / 'reports' / f).exists()]
fig, axes = plt.subplots(1, len(imgs), figsize=(6 * len(imgs), 6))
for ax, (name, path) in zip(np.atleast_1d(axes), imgs):
    ax.imshow(mpimg.imread(path)); ax.axis('off'); ax.set_title(name)
plt.tight_layout(); plt.show()

## Takeaways (interview-ready)
- **Classic ML wins on GTZAN.** Hand-crafted features (MFCC = timbre, chroma =
  harmony, etc.) carry most of the genre signal; with only ~700 training clips a
  from-scratch CNN can't out-learn them.
- **It's also ~80x cheaper at inference** (sub-ms vs ~19ms/clip on CPU).
- **`rock` is everyone's worst class** — GTZAN rock overlaps heavily with other
  guitar-driven genres; the CNN collapses on it (F1≈0.12).
- **When would the CNN win?** More data (FMA), longer training, or transfer
  learning from a pretrained audio model — all out of scope for this 4GB,
  small-data benchmark, but the right next step to mention.